# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd

In [2]:
# cargar archivos
url = 'https://practicum-content.s3.amazonaws.com/datasets/'
orders = pd.read_csv(url+'rappiplus_orders_raw.csv')
catalog = pd.read_csv(url+'rappiplus_catalog.csv')
marketing = pd.read_csv(url+'rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets
# tu código aquí
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


## Orders 
present 12 columns with following caracteristics
- id_pedido,id_usuario represent key columna for identification
- fecha_fora_pedido: date columna
- pais,dispositivo,fuente_referencia, categoria_producto: categorical
- nombre_producto: descriptive and also categorical value
- cantidad, precio_unitario, monto_descuento, monto_total: numeric values

its important recognize although column `fecha_hora_pedido` says includes hour the value doesnt include the hour at all

In [4]:
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


## Marketing 
table present 5 colums with the following characteristics
- fecha: date
- pais, canal: categorical
- id_campaña: key relation value
- gasto: numeric value

In [5]:
catalog.head(6)

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith


## Catalogo
table present 4 colums with the following characteristics
- Nombre_producto: Descriptive Categoryc
- Categoria_producto: categorical
- costo_unitario: numeric value
- proveedor: categorical

# 

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

### Exploratory analysis for `orders`

In [6]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


Its necesary transform the column `fecha_hora_pedido` to datetime and make sure it is not missing values

In [7]:
orders_clean = orders.copy() #creating the copy
# Specify the format (faster and avoids ambiguity)
orders["fecha_hora_pedido"] = pd.to_datetime(orders["fecha_hora_pedido"], format="%Y-%m-%d")
orders_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


Now i make a stadistical resume to numeric columns to analysis the values

In [8]:
columns_number_orders = ["cantidad","precio_unitario","monto_descuento","monto_total"]

orders[columns_number_orders].describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


Here it can be identified indeed there are 50 nan values in `cantidad`,`precio_unitario` and `monto_descuento`.

also its been identified negative values in `cantidad` and `monto_total` that could represents strange values

In [9]:
# Count negative quantities
print("Negative quantities:", (orders_clean['cantidad'] < 0).sum())

# Inspect affected records

display(
    orders_clean.loc[
        orders_clean['cantidad'] < 0,
        ['id_pedido', 'cantidad', 'precio_unitario',
         'monto_descuento', 'monto_total']
    ].head(20)
)


Negative quantities: 4


,id_pedido,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,-2.0,101.31,10.0,-192.62
267,order_267,-1.0,43.50,5.0,-38.50
268,order_268,-1.0,497.65,5.0,-492.65
269,order_269,-1.0,423.53,0.0,-423.53


`monto_total` was affected due of `cantidad` have negative values, so it will require transform the negative values to positive and recalculate `monto_final`

In [10]:
orders_clean['cantidad'] = orders_clean['cantidad'].abs()

orders_clean['monto_total'] = (
    orders_clean['cantidad'] * orders_clean['precio_unitario']
    - orders_clean['monto_descuento']
)

orders_clean.loc[267] #quick verification of the calculation are correct


id_pedido                   order_267
id_usuario                  user_1087
fecha_hora_pedido          2025-05-07
pais                              NaN
dispositivo                   desktop
fuente_referencia              social
nombre_producto       Phone-Pro-128GB
categoria_producto        Electronica
cantidad                          1.0
precio_unitario                  43.5
monto_descuento                   5.0
monto_total                      38.5
Name: 267, dtype: object

Confirmation for there id no more records with nan values in `cantidad`,`precio_unitario` and `monto_descuento` 

Now ill do a verification for duplicated records

In [11]:
orders.duplicated().sum()

100

Identify 100 duplicates

In [12]:
duplicates = orders_clean[
    orders_clean.duplicated(keep=False)
]

display(duplicates)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
734,order_734,user_6347,2025-05-02,Colombia,desktop,organic,Blender-XL-Red,Hogar,1.0,58.00,0.0,58.00
812,order_812,user_1530,2025-03-31,Argentina,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,167.32,10.0,157.32
974,order_974,user_3262,2025-05-04,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,2.0,268.58,5.0,532.16
1361,order_1361,user_507,2025-06-27,Mexico,desktop,paid_search,Laptop-Gaming-16GB,Electronica,2.0,239.61,0.0,479.22
1735,order_1735,user_728,2025-05-13,Mexico,mobile,social,Tablet-Standard-64GB,Electronica,2.0,419.04,0.0,838.08
...,...,...,...,...,...,...,...,...,...,...,...,...
25095,order_3913,user_380,2025-02-18,Argentina,desktop,paid_search,Phone-Pro-128GB,Electronica,1.0,82.28,0.0,82.28
25096,order_23405,user_7833,2025-04-04,Colombia,mobile,paid_search,Phone-Pro-128GB,Electronica,2.0,99.25,5.0,193.50
25097,order_5615,user_5417,2025-05-13,Colombia,desktop,social,Blender-XL-Red,Hogar,2.0,450.35,5.0,895.70
25098,order_812,user_1530,2025-03-31,Argentina,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,167.32,10.0,157.32


In [13]:
# Display only the duplicate copies, excluding the first occurrence
display(orders_clean[orders_clean.duplicated()])

# Count all rows involved in duplicates
print("Rows involved in duplicates:", len(duplicates))

# Count duplicate copies only
print("Duplicate copies:", orders_clean.duplicated().sum())

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
25000,order_22936,user_4028,2025-06-30,Argentina,desktop,social,Vacuum-Pro-Black,Hogar,2.0,41.91,10.0,73.82
25001,order_13710,user_4466,2025-03-20,Mexico,desktop,social,Jacket-Winter-M,Moda,2.0,483.32,0.0,966.64
25002,order_14562,user_6590,2025-03-23,Mexico,mobile,paid_search,Tablet-Standard-64GB,Electronica,1.0,481.96,5.0,476.96
25003,order_11537,user_3115,2025-03-08,Argentina,desktop,organic,Vacuum-Pro-Black,Hogar,1.0,147.20,5.0,142.20
25004,order_4533,user_7944,2025-02-16,Argentina,mobile,organic,Blender-XL-Red,Hogar,1.0,176.91,5.0,171.91
...,...,...,...,...,...,...,...,...,...,...,...,...
25095,order_3913,user_380,2025-02-18,Argentina,desktop,paid_search,Phone-Pro-128GB,Electronica,1.0,82.28,0.0,82.28
25096,order_23405,user_7833,2025-04-04,Colombia,mobile,paid_search,Phone-Pro-128GB,Electronica,2.0,99.25,5.0,193.50
25097,order_5615,user_5417,2025-05-13,Colombia,desktop,social,Blender-XL-Red,Hogar,2.0,450.35,5.0,895.70
25098,order_812,user_1530,2025-03-31,Argentina,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,167.32,10.0,157.32


Rows involved in duplicates: 200
Duplicate copies: 100


In [14]:
# Check duplicated order IDs
display(
    orders_clean[
        orders_clean.duplicated(subset=['id_pedido'], keep=False)
    ].sort_values('id_pedido')
)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
25023,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.68
10082,order_10082,user_690,2025-02-17,Argentina,desktop,social,Sneakers-Urban-42,Moda,2.0,221.34,0.0,442.68
25037,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
10709,order_10709,user_6783,2025-02-16,Colombia,mobile,organic,Jacket-Winter-M,Moda,1.0,170.10,10.0,160.10
25065,order_10829,user_7697,2025-01-23,Argentina,mobile,social,Phone-Pro-128GB,Electronica,2.0,115.54,0.0,231.08
...,...,...,...,...,...,...,...,...,...,...,...,...
25048,order_8326,user_6177,2025-06-14,Argentina,mobile,organic,Vacuum-Pro-Black,Hogar,1.0,457.53,0.0,457.53
25043,order_8414,user_4073,2025-05-10,Colombia,mobile,social,Jacket-Winter-M,Moda,2.0,144.35,10.0,278.70
8414,order_8414,user_4073,2025-05-10,Colombia,mobile,social,Jacket-Winter-M,Moda,2.0,144.35,10.0,278.70
25056,order_974,user_3262,2025-05-04,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,2.0,268.58,5.0,532.16


i identified the duplicated records present the exact same data including `id_pedido` in this case it requires drop this values because it could cause a bias in the final analysis

In [15]:
orders_clean = orders_clean.drop_duplicates().copy()

print("Remaining rows:", len(orders_clean))
print("Duplicate rows remaining:", orders_clean.duplicated().sum())

Remaining rows: 25000
Duplicate rows remaining: 0


now in order to find some patter to imputate the records with the null values in `cantidad`,`precio_unitario` and `monto_descuento` before of decide to drop them

In [16]:

cols = ['cantidad', 'precio_unitario', 'monto_descuento']

missing_rows_orders = orders[orders[cols].isnull().any(axis=1)]

missing_rows_orders.head(5)



,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
74,order_74,user_6172,2025-01-15,Argentina,desktop,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,595.85
75,order_75,user_6588,2025-06-19,Colombia,mobile,organic,Sneakers-Urban-42,NaN,NaN,NaN,NaN,458.15
76,order_76,user_3193,2025-03-17,Argentina,mobile,paid_search,Laptop-Gaming-16GB,NaN,NaN,NaN,NaN,319.75
77,order_77,user_775,2025-06-24,Argentina,desktop,social,Vacuum-Pro-Black,NaN,NaN,NaN,NaN,227.55
78,order_78,user_2702,2025-03-19,Argentina,desktop,paid_search,Phone-Pro-128GB,NaN,NaN,NaN,NaN,432.39


In [17]:
orders[cols].isnull().value_counts().rename("Number of rows").to_frame()



,,,Number of rows
cantidad,precio_unitario,monto_descuento,
False,False,False,25050
True,True,True,50


There is no values that i could use in order to recalculate the missing monetary values.

for this reason i decided to measure the impact of the dropping records with the missing values over the dataset and the final analysis

In [18]:
cols = ['cantidad', 'precio_unitario', 'monto_descuento']

# Identify rows with all three values missing
mask = orders[cols].isna().all(axis=1)

# Compare dataset before and after removal
total_rows = len(orders)
affected_rows = mask.sum()
remaining_rows = total_rows - affected_rows

print(f"Original rows: {total_rows:,}")
print(f"Rows affected: {affected_rows:,}")
print(f"Rows remaining: {remaining_rows:,}")
print(f"Percentage removed: {affected_rows / total_rows * 100:.2f}%")

Original rows: 25,100
Rows affected: 50
Rows remaining: 25,050
Percentage removed: 0.20%


In [19]:
revenue_before = orders['monto_total'].sum()
revenue_affected = orders.loc[mask, 'monto_total'].sum()
revenue_after = orders.loc[~mask, 'monto_total'].sum()

print(f"Revenue before: {revenue_before:,.2f}")
print(f"Revenue in affected rows: {revenue_affected:,.2f}")
print(f"Revenue after removal: {revenue_after:,.2f}")

print(
    f"Revenue lost: "
    f"{revenue_affected / revenue_before * 100:.2f}%"
    if revenue_before != 0 else "Revenue before is zero"
)

Revenue before: 52,024,257.56
Revenue in affected rows: 21,628.00
Revenue after removal: 52,002,629.56
Revenue lost: 0.04%


In [20]:
comparison = orders.assign(
    data_group=mask.map({
        True: 'Missing quantity, price and discount',
        False: 'Complete quantity, price and discount'
    })
)

summary = comparison.groupby('data_group').agg(
    orders=('id_pedido', 'count'),
    total_revenue=('monto_total', 'sum'),
    average_order_value=('monto_total', 'mean'),
    median_order_value=('monto_total', 'median')
)

display(summary.round(2))

,orders,total_revenue,average_order_value,median_order_value
data_group,,,,
"Complete quantity, price and discount",25050,52002629.56,2075.95,341.48
"Missing quantity, price and discount",50,21628.00,432.56,437.11


the stadistical analysis shows the impact on eliminate the record with the nan values in the columns `cantidad`,`precio_unitario`,`monto_descuento` represent a loss of 0.2% of the values
and also represent a loss of 0.04% for the total of `monto_total`


Before of eliminate the nan records i will make a final comparassion on the change after and before of the drop

In [21]:
orders_clean = orders_clean.dropna(subset=cols, how='all')

# Imputation of negatives values
orders_clean['cantidad'] = orders_clean['cantidad'].abs()
orders_clean['monto_total'] = (
    orders_clean['cantidad'] * orders_clean['precio_unitario']
    - orders_clean['monto_descuento']
) 

print("Original dataset:", orders.shape)
print("Cleaned dataset:", orders_clean.shape)
print("Rows removed:", len(orders) - len(orders_clean))

comparison = pd.concat(
    [
        orders['monto_total'].describe().rename('Before cleaning'),
        orders_clean['monto_total'].describe().rename('After cleaning')
    ],
    axis=1
)

display(comparison)

Original dataset: (25100, 12)
Cleaned dataset: (24950, 12)
Rows removed: 150


,Before cleaning,After cleaning
count,2.510000e+04,2.495000e+04
mean,2.072680e+03,2.082891e+03
std,9.894995e+04,9.924687e+04
min,-4.926500e+02,5.240000e+00
25%,1.805075e+02,1.806400e+02
50%,3.417500e+02,3.416100e+02
75%,5.185800e+02,5.183500e+02
max,8.840200e+06,8.840200e+06


In [22]:
orders_clean = orders_clean.dropna(subset=cols, how='all')

# Imputation of negatives values
orders_clean['cantidad'] = orders_clean['cantidad'].abs()
orders_clean['monto_total'] = (
    orders_clean['cantidad'] * orders_clean['precio_unitario']
    - orders_clean['monto_descuento']
) 

print("Original dataset:", orders.shape)
print("Cleaned dataset:", orders_clean.shape)
print("Rows removed:", len(orders) - len(orders_clean))

comparison = pd.concat(
    [
        orders['monto_total'].describe().rename('Before cleaning'),
        orders_clean['monto_total'].describe().rename('After cleaning')
    ],
    axis=1
)

display(comparison)

Original dataset: (25100, 12)
Cleaned dataset: (24950, 12)
Rows removed: 150


,Before cleaning,After cleaning
count,2.510000e+04,2.495000e+04
mean,2.072680e+03,2.082891e+03
std,9.894995e+04,9.924687e+04
min,-4.926500e+02,5.240000e+00
25%,1.805075e+02,1.806400e+02
50%,3.417500e+02,3.416100e+02
75%,5.185800e+02,5.183500e+02
max,8.840200e+06,8.840200e+06


comparating values before and after of droping nan values, theres no significant loss

Now i will look up for some pattern in order to imputate `pais`,`dispositivo`,`fuente referencia`,`nombre_producto`,`categoria_producto` before of onsiderate eliminate

first determinate for `pais`

I will search for records that share the same `id_client` and actually present a value in `pais` to imputate the missing values

In [23]:
missing_country_records = orders_clean[
    orders_clean['pais'].isna()
].copy()

known_countries = (
    orders_clean.dropna(subset=['pais'])
    [['id_usuario', 'pais']]
    .drop_duplicates()
)

country_matches = missing_country_records.merge(
    known_countries,
    on='id_usuario',
    how='left',
    suffixes=('_missing', '_known')
)

display(country_matches[
    ['id_usuario', 'pais_missing', 'pais_known']
])

,id_usuario,pais_missing,pais_known
0,user_6671,NaN,Mexico
1,user_5263,NaN,Argentina
2,user_4952,NaN,Mexico
3,user_4952,NaN,mexico
4,user_2619,NaN,Mexico
...,...,...,...
381,user_2674,NaN,Colombia
382,user_2248,NaN,argentina
383,user_2248,NaN,Argentina
384,user_5872,NaN,Mexico


In [24]:
orders_clean["pais"] = orders_clean["pais"].fillna(
    orders_clean.groupby("id_usuario")["pais"].transform("first")
)

In [25]:

print("Missing countries remaining:",
      orders_clean['pais'].isna().sum())

print("Total records:", len(orders_clean))

display(
    orders_clean.loc[
        orders_clean['pais'].notna(),
        ['id_usuario', 'pais']
    ].head(10)
)


Missing countries remaining: 19
Total records: 24950


,id_usuario,pais
0,user_6993,Argentina
1,user_1329,Mexico
2,user_3194,Argentina
3,user_4510,Colombia
4,user_5044,Argentina
5,user_2792,mexico
6,user_2802,Colombia
7,user_1083,Mexico
8,user_2352,Argentina
9,user_7001,Mexico


In [26]:
missing_before = orders['pais'].isna().sum()
missing_after = orders_clean['pais'].isna().sum()
recovered = missing_before - missing_after

print("Missing before:", missing_before)
print("Successfully imputed:", recovered)
print("Still missing:", missing_after)
print(f"Recovery rate: {recovered / missing_before * 100:.2f}%"
      if missing_before else "No missing values to recover")

Missing before: 300
Successfully imputed: 281
Still missing: 19
Recovery rate: 93.67%


In [27]:
nulos = orders_clean[orders_clean["pais"].isna()]
print("Null values non imputables:", nulos["id_usuario"].nunique())   # how many different clients
print("Total revenue concentrated in those records: ",nulos["monto_total"].sum())      # how much revenue is involved
print("% revenue it represents:",(nulos["monto_total"].sum() / orders["monto_total"].sum())*100)

Null values non imputables: 18
Total revenue concentrated in those records:  6813.459999999999
% revenue it represents: 0.01309669819341868


A final comprobation provees there is no more imputable records with `pais` missing, so ill drop them considerating this quantity dont represent a significant value to the whole data set

In [28]:
orders_clean = orders_clean.dropna(subset=["pais"])

print("Actual record with Null values: ",orders_clean["pais"].isna().sum())  # should be 0
print("Restant values after dropping: ",len(orders_clean))

orders_clean = orders_clean.reset_index(drop=True)

Actual record with Null values:  0
Restant values after dropping:  24931


Imputation for `dispositivo`

firt i will look up for record that doesnt hava category and name missing in the same record

In [29]:
products_missing_category = orders_clean[
    orders_clean['nombre_producto'].notna()
    & orders_clean['categoria_producto'].isna()
]

display(
    products_missing_category[
        ['nombre_producto', 'categoria_producto']
    ].drop_duplicates()
)

,nombre_producto,categoria_producto


There is no record that could use for imputate, then i will measure the impact before removing records

In [30]:
cols = ['nombre_producto', 'categoria_producto']

orders_products_clean = orders_clean.dropna(
    subset=cols
).copy()

comparison = pd.DataFrame({
    'Before cleaning': orders_clean['monto_total'].describe(),
    'After cleaning': orders_products_clean['monto_total'].describe()
})

comparison['Difference'] = (
    comparison['After cleaning'] - comparison['Before cleaning']
)

display(comparison.round(2))

,Before cleaning,After cleaning,Difference
count,24931.00,24901.00,-30.00
mean,2084.21,2086.22,2.02
std,99284.67,99344.45,59.77
min,5.24,5.24,0.00
25%,180.66,180.64,-0.02
50%,341.62,341.62,0.00
75%,518.43,518.20,-0.23
max,8840200.00,8840200.00,0.00


In [31]:
rows_before = len(orders_clean)
rows_after = len(orders_products_clean)
rows_removed = rows_before - rows_after

revenue_before = orders_clean['monto_total'].sum()
revenue_after = orders_products_clean['monto_total'].sum()
revenue_lost = revenue_before - revenue_after

print(f"Rows removed: {rows_removed:,}")
print(f"Percentage of rows removed: {rows_removed / rows_before * 100:.2f}%")

print(f"\nRevenue before: {revenue_before:,.2f}")
print(f"Revenue after: {revenue_after:,.2f}")
print(f"Revenue lost: {revenue_lost:,.2f}")

if revenue_before != 0:
    print(
        f"Percentage of revenue lost: "
        f"{revenue_lost / revenue_before * 100:.2f}%"
    )
orders_cleaned = orders_products_clean

Rows removed: 30
Percentage of rows removed: 0.12%

Revenue before: 51,961,316.21
Revenue after: 51,949,053.62
Revenue lost: 12,262.59
Percentage of revenue lost: 0.02%


The difference dont represent a relevant loss in the whole dataset

Now analysis for column `fuente_referencia`

In [32]:

orders_no_source = orders_clean.dropna(
    subset=['fuente_referencia']
).copy()

rows_before = len(orders_clean)
rows_after = len(orders_no_source)

revenue_before = orders_clean['monto_total'].sum()
revenue_after = orders_no_source['monto_total'].sum()

print(f"Rows removed: {rows_before - rows_after:,}")
print(
    f"Percentage of rows removed: "
    f"{(rows_before - rows_after) / rows_before * 100:.2f}%"
)

print(f"\nRevenue before: {revenue_before:,.2f}")
print(f"Revenue after: {revenue_after:,.2f}")

if revenue_before != 0:
    print(
        f"Revenue lost: "
        f"{(revenue_before - revenue_after) / revenue_before * 100:.2f}%"
    )

order_clean = orders_no_source


Rows removed: 30
Percentage of rows removed: 0.12%

Revenue before: 51,961,316.21
Revenue after: 51,949,053.62
Revenue lost: 0.02%


I considerate them drop because his impact to the analysis doent represent a relevant value

Now analysis for column `dispositivo`

In [33]:
dispositivo_source = orders_clean.dropna(
    subset=['dispositivo']
).copy()

rows_before = len(orders_clean)
rows_after = len(dispositivo_source)

revenue_before = orders_clean['monto_total'].sum()
revenue_after = dispositivo_source['monto_total'].sum()

print(f"Rows removed: {rows_before - rows_after:,}")
print(
    f"Percentage of rows removed: "
    f"{(rows_before - rows_after) / rows_before * 100:.2f}%"
)

print(f"\nRevenue before: {revenue_before:,.2f}")
print(f"Revenue after: {revenue_after:,.2f}")

if revenue_before != 0:
    print(
        f"Revenue lost: "
        f"{(revenue_before - revenue_after) / revenue_before * 100:.2f}%"
    )
    
orders_clean = dispositivo_source

Rows removed: 20
Percentage of rows removed: 0.08%

Revenue before: 51,961,316.21
Revenue after: 51,954,960.46
Revenue lost: 0.01%


For the final i will make a quick analysis for the final dataset of orders cleaned

In [34]:
orders_clean.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,24911.000000,24911.000000,24911.000000,2.491100e+04
mean,7.124563,259.375881,4.501224,2.085623e+03
std,297.102167,138.672404,5.223729,9.932451e+04
min,1.000000,20.030000,0.000000,5.240000e+00
25%,1.000000,138.555000,0.000000,1.807650e+02
50%,2.000000,258.790000,0.000000,3.417800e+02
75%,2.000000,380.315000,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [35]:
orders_clean[["pais","dispositivo","fuente_referencia","nombre_producto","categoria_producto"]].describe()

,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto
count,24911,24911,24881,24881,24881
unique,6,2,3,7,3
top,Mexico,desktop,social,Blender-XL-Red,Hogar
freq,7543,12676,8370,4169,8331


at this moment i dont considerate there is no neccesary any more clean for this dataset

### Now i will continue with marketing

firts ill use statistical resume for marketing

In [36]:
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [37]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


For this data set its necessary transform the column fecha to date

In [38]:
marketing_clean = marketing.copy() #creating the copy
marketing_clean["fecha"] = pd.to_datetime(marketing["fecha"], format="%Y-%m-%d")
marketing_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


In [39]:
marketing.describe()

,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


In [40]:
marketing[["fecha","pais","id_campaña","canal"]].describe()

,fecha,pais,id_campaña,canal
count,1620,1620,1620,1519
unique,180,3,9,3
top,2025-06-05,Argentina,paid_search_Colombia,paid_search
freq,9,540,180,507


In [41]:
marketing.isna().sum()

fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64

In [42]:
marketing[marketing["canal"].isna()]

,fecha,pais,id_campaña,canal,gasto
98,2025-01-11,Argentina,social_Argentina,NaN,849.70
99,2025-01-12,Mexico,organic_Mexico,NaN,2033.56
100,2025-01-12,Mexico,paid_search_Mexico,NaN,1260.65
101,2025-01-12,Mexico,social_Mexico,NaN,1660.90
102,2025-01-12,Colombia,organic_Colombia,NaN,1819.27
...,...,...,...,...,...
194,2025-01-22,Colombia,social_Colombia,NaN,772.46
195,2025-01-22,Argentina,organic_Argentina,NaN,1562.61
196,2025-01-22,Argentina,paid_search_Argentina,NaN,644.34
197,2025-01-22,Argentina,social_Argentina,NaN,2544.66


i will calculate the impact of removing the records with canal null

In [43]:
marketing_no_nulls = marketing.dropna(subset=['canal']).copy()

# Compare row counts
rows_before = len(marketing)
rows_after = len(marketing_no_nulls)
rows_removed = rows_before - rows_after

print(f"Rows before: {rows_before:,}")
print(f"Rows after: {rows_after:,}")
print(f"Rows removed: {rows_removed:,}")
print(f"Percentage removed: {rows_removed / rows_before * 100:.2f}%")

Rows before: 1,620
Rows after: 1,519
Rows removed: 101
Percentage removed: 6.23%


The records that represent missing values in canal its really high, so for this dataset i decided to imputate with value Other

In [44]:
marketing_clean['canal'] = marketing['canal'].fillna('Other')

In [45]:
marketing_clean.isna().sum()

fecha         0
pais          0
id_campaña    0
canal         0
gasto         0
dtype: int64

for this dataset i consider there is no more neccesary imputations or fix

### analysis for `catalog`

In [46]:
catalog.head()

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [47]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [48]:
catalog.isna().sum()

nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64

In [49]:
catalog.describe()

,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


In [50]:
catalog[["nombre_producto","categoria_producto","proveedor"]].describe()

,nombre_producto,categoria_producto,proveedor
count,7,7,7
unique,7,3,7
top,Tablet-Standard-64GB,Electrónica,Bowers LLC
freq,1,3,1


for catalog due for his short dataset no present missing values or any error that could may require imputation or fixment

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [51]:
# exportar datasets
orders_clean.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing_clean.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

### part 1 - Key indicator results

In [52]:
total_revenue = orders_clean["monto_total"].sum() # Total Revenue´
print(f"Total Revenue: ${total_revenue:,.2f}")

Total Revenue: $51,954,960.46


In order to can calculate the total cost i cross the tables catalog and order to multiply the quantity of each order with the real price

In [53]:
# Merge orders with the product catalog
orders_cost = orders_clean.merge(
    catalog[['nombre_producto', 'costo_unitario']],
    on='nombre_producto',
    how='left'
)

# Calculate cost for each order
orders_cost['costo_total_producto'] = (
    orders_cost['cantidad'] * orders_cost['costo_unitario']
)

# Calculate total cost
total_cost = orders_cost['costo_total_producto'].sum()

print(f"Total Cost: ${total_cost:,.2f}")

Total Cost: $43,119,586.49


I make a verification of the orders where cost is greather than the revenue

In [54]:
records_overcost = orders_cost[orders_cost['costo_total_producto'] > orders_cost["monto_total"]]
print("Total cost of records with overcorst: ", records_overcost["costo_total_producto"].sum())
print("Total revenue of records with overcost: ", records_overcost["monto_total"].sum())


Total cost of records with overcorst:  12670755.479999999
Total revenue of records with overcost:  7274221.16


In [55]:
p_over_cost = (records_overcost["costo_total_producto"].sum() / total_cost) * 100
print(f"how much of the total cost represent the records with overcost {p_over_cost:,.2f}%")

how much of the total cost represent the records with overcost 29.39%


In [56]:
total_cost_marketing = marketing_clean["gasto"].sum() #total cost
print(f"Total marketing cost: ${total_cost_marketing:,.2f}")

Total marketing cost: $2,871,843.53


In [57]:
profit = total_revenue - (total_cost + total_cost_marketing)
print(profit)
print(f"Percentage of profits {(profit / total_revenue) * 100}")


5963530.439999998
Percentage of profits 11.478269615066507


## KPI

In [58]:
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Cost: ${total_cost:,.2f}")
print(f"Total marketing cost: ${total_cost_marketing:,.2f}")
print(f"Total profits: ${profit:,.2f}")

Total Revenue: $51,954,960.46
Total Cost: $43,119,586.49
Total marketing cost: $2,871,843.53
Total profits: $5,963,530.44


### part 2 - sells behavior

In [59]:
avg_order = orders_clean["monto_total"].mean()
print(f"Averague ticket per order {avg_order}")

Averague ticket per order 2085.623237124162


In [60]:
avg_quantity_order = orders_clean["cantidad"].mean()
print(f"Averague quantity per order {avg_quantity_order}")

Averague quantity per order 7.124563445867287


In [61]:
sales_by_product = (
    orders_clean.groupby('nombre_producto')['cantidad']
    .sum()
    .sort_values(ascending=False)
)

most_sold_product = sales_by_product.index[0]
total_units_sold = sales_by_product.iloc[0]

print("Most Sold Product:", most_sold_product)
print(f"Total Units Sold: {total_units_sold:,.0f}")

Most Sold Product: Laptop-Gaming-16GB
Total Units Sold: 144,194


In [62]:
cost_by_channel = (
    marketing_clean.groupby('canal')['gasto']
    .sum()
)

print("Marketing cost by channel")
display(cost_by_channel)


Marketing cost by channel


canal
Other          177179.10
organic        913533.01
paid_search    863088.21
social         918043.21
Name: gasto, dtype: float64

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [63]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [64]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [65]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
WITH funnel AS (
    SELECT
        COUNT(DISTINCT CASE
            WHEN nombre_evento = 'first_visit'
            THEN id_usuario
        END) AS first_visit,

        COUNT(DISTINCT CASE
            WHEN nombre_evento = 'add_to_cart'
            THEN id_usuario
        END) AS add_to_cart,

        COUNT(DISTINCT CASE
            WHEN nombre_evento = 'begin_checkout'
            THEN id_usuario
        END) AS begin_checkout,

        COUNT(DISTINCT CASE
            WHEN nombre_evento = 'add_payment_info'
            THEN id_usuario
        END) AS add_payment_info,

        COUNT(DISTINCT CASE
            WHEN nombre_evento = 'purchase'
            THEN id_usuario
        END) AS purchase

    FROM events
)

SELECT
    'First Visit' AS etapa,
    first_visit AS usuarios,
    100.0 AS conversion_pct
FROM funnel

UNION ALL

SELECT
    'Add to Cart',
    add_to_cart,
    ROUND(100.0 * add_to_cart / NULLIF(first_visit, 0), 2)
FROM funnel

UNION ALL

SELECT
    'Begin Checkout',
    begin_checkout,
    ROUND(100.0 * begin_checkout / NULLIF(first_visit, 0), 2)
FROM funnel

UNION ALL

SELECT
    'Add Payment Info',
    add_payment_info,
    ROUND(100.0 * add_payment_info / NULLIF(first_visit, 0), 2)
FROM funnel

UNION ALL

SELECT
    'Purchase',
    purchase,
    ROUND(100.0 * purchase / NULLIF(first_visit, 0), 2)
FROM funnel;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,etapa,usuarios,conversion_pct
0,First Visit,7796,100.00
1,Add to Cart,7634,97.92
2,Begin Checkout,7208,92.46
3,Add Payment Info,6250,80.17
4,Purchase,6240,80.04


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 
---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [66]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [70]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT * FROM user_activity
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(10)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0
4,user_1,2025-01-14,7,0
5,user_1,2025-01-21,14,0
6,user_1,2025-01-28,21,1
7,user_1,2025-02-04,28,0
8,user_2,2025-03-19,7,0
9,user_2,2025-03-26,14,1


In [141]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohort AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', fecha_registro::timestamp) AS cohort_month
    FROM users
),
retention AS (
SELECT 
    cohort_month,
    COUNT(*) AS registered_clients,
    COUNT(CASE WHEN dias_despues_registro = 7 AND activo = 1 THEN 1 END) AS retention_w1,
    COUNT(CASE WHEN dias_despues_registro = 14 AND activo = 1 THEN 1 END) AS retention_w2,
    COUNT(CASE WHEN dias_despues_registro = 21 AND activo = 1 THEN 1 END) AS retention_w3
FROM 
    cohort
LEFT JOIN user_activity ua ON ua.id_usuario = cohort.id_usuario
GROUP BY 
    cohort_month
)

SELECT 
    TO_CHAR(cohort_month::timestamp,'YYYY-MM') AS cohorte,
    registered_clients,
    ROUND((retention_w1::numeric / registered_clients)*100,2) AS Week1,
    retention_w1,
    ROUND((retention_w2::numeric / registered_clients)*100,2) AS Week2,
    retention_w2,
    ROUND((retention_w3::numeric / registered_clients)*100,2) AS Week3,
    retention_w3
FROM retention
'''



# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)

cohorte_final

,cohorte,registered_clients,week1,retention_w1,week2,retention_w2,week3,retention_w3
0,2025-02,5776,10.58,611,10.54,609,10.99,635
1,2025-04,6424,10.59,680,10.85,697,10.32,663
2,2025-03,6544,10.35,677,10.77,705,10.54,690
3,2025-01,6508,10.71,697,10.26,668,10.08,656
4,2025-05,6748,10.30,695,10.02,676,10.46,706


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  


---

Hipótesis estadística
   - **H₀ (Hipótesis nula):** There is no statistically significant difference in the conversion rate between the control group and the treatment group.
   - **H₁ (Hipótesis alternativa):** There is statistically significant difference in the conversion rate between the control group and the treatment group
   
**Test estadístico:** Two-Proportion Z-Test, with bilateral contrast.

**Nivel de significancia alpha:**  (\alpha = 0.05) (5%).

**Criterio de decisión:** If p-value is less than 0.05, the null hipotesis is rejected, in the otherwise, doesnt find statistically evidence to rejected.


In [147]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep

experiment = pd.read_csv(url+'experiment_checkout_ui.csv')

experiment


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12
...,...,...,...,...,...,...,...
9995,exp_user_9995,tratamiento,0,mobile,Mexico,99.61,2025-01-27
9996,exp_user_9996,tratamiento,0,desktop,Argentina,209.82,2025-01-03
9997,exp_user_9997,tratamiento,0,mobile,Argentina,271.15,2025-02-26
9998,exp_user_9998,control,0,mobile,Colombia,216.11,2025-02-03


In [148]:
# Keep only valid experimental groups and binary outcomes
experiment_ab = experiment[
    experiment["variante"].isin(["control", "tratamiento"])
    & experiment["convirtio"].isin([0, 1])
].copy()

experiment_ab["convirtio"] = experiment_ab["convirtio"].astype(int)

# Check the sample sizes and outcome distribution
print(pd.crosstab(experiment_ab["variante"], experiment_ab["convirtio"]))


convirtio       0    1
variante              
control      4186  779
tratamiento  4215  820


In [151]:
results = (
    experiment_ab.groupby("variante")["convirtio"]
    .agg(["sum", "count", "mean"])
)

results.columns = ["conversiones", "usuarios", "tasa_conversion"]

results["tasa_conversion"] = results["tasa_conversion"] * 100

display(results)

,conversiones,usuarios,tasa_conversion
variante,,,
control,779,4965,15.689829
tratamiento,820,5035,16.285998


In [152]:
# Extract treatment and control results
treatment = results.loc["tratamiento"]
control = results.loc["control"]

success_t = int(treatment["conversiones"])
n_t = int(treatment["usuarios"])

success_c = int(control["conversiones"])
n_c = int(control["usuarios"])

rate_t = success_t / n_t
rate_c = success_c / n_c

# Absolute difference in percentage points
absolute_change = (rate_t - rate_c) * 100

# Relative lift in percentage
relative_lift = (
    (rate_t - rate_c) / rate_c * 100
    if rate_c > 0 else np.nan
)

# Two-proportion z-test
z_stat, p_value = proportions_ztest(
    count=[success_t, success_c],
    nobs=[n_t, n_c],
    alternative="two-sided"
)

# 95% confidence interval for treatment - control
ci_low, ci_high = confint_proportions_2indep(
    count1=success_t,
    nobs1=n_t,
    count2=success_c,
    nobs2=n_c,
    method="newcomb",
    alpha=0.05
)

print(f"Control rate:       {rate_c:.2%}")
print(f"Treatment rate:     {rate_t:.2%}")
print(f"Absolute change:    {absolute_change:+.2f} percentage points")
print(f"Relative lift:      {relative_lift:+.2f}%")
print(f"95% CI:             [{ci_low*100:.2f}, {ci_high*100:.2f}] percentage points")
print(f"P-value:            {p_value:.4f}")

if p_value < 0.05:
    print("Conclusion: Statistically significant difference.")
else:
    print("Conclusion: No statistically significant difference detected.")

Control rate:       15.69%
Treatment rate:     16.29%
Absolute change:    +0.60 percentage points
Relative lift:      +3.80%
95% CI:             [-0.84, 2.03] percentage points
P-value:            0.4161
Conclusion: No statistically significant difference detected.


###

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive